In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from riemannian import *

## Flat metric

In [ ]:
"""Test flat metric (g₁₁ = 1)."""
x = symbols('x', real=True)
metric = Metric(1, x)

# Christoffel should be zero
assert metric.christoffel_sym == 0

# Geodesics should be straight lines
traj = geodesic_solver(metric, 0.0, 1.0, (0, 5), n_steps=100)
expected = np.linspace(0, 5, 100)
assert np.allclose(traj['x'], expected, rtol=1e-2)

# Volume should equal length
vol = metric.riemannian_volume((0, 1), method='symbolic')
assert vol == 1

print("✓ Flat metric test passed")

## Poincaré half-plane

In [ ]:
"""Test hyperbolic metric (Poincaré half-plane)."""
x = symbols('x', real=True, positive=True)
metric = Metric(1/x**2, x)

# Check Christoffel symbol
expected_gamma = -1/x
assert simplify(metric.christoffel_sym - expected_gamma) == 0

# Check arc length
vol = metric.riemannian_volume((1, 2), method='numerical')
expected = np.log(2)  # ∫₁² dx/x = log(2)
assert np.isclose(vol, expected, rtol=1e-3)

print("✓ Hyperbolic metric test passed")

## Metric extraction from Hamiltonian

In [ ]:
"""Test metric extraction from Hamiltonian."""
x, p = symbols('x p', real=True)

# Harmonic oscillator with varying mass
H = p**2 / (2*x**2) + x**2 / 2
metric = Metric.from_hamiltonian(H, (x,), (p,))

# Should extract g = x²
assert simplify(metric.g_expr - x**2) == 0

print("✓ Hamiltonian extraction test passed")

## Parallel Transport on the Cone

In [ ]:
import numpy as np
import sympy as sp
from sympy import symbols, simplify, Matrix
import matplotlib.pyplot as plt
from riemannian import Metric, geodesic_solver, parallel_transport
from riemannian import visualize_geodesics, christoffel

# Define the 1D cone metric g = x² (a simple but non‑trivial metric)
x = symbols('x', real=True, positive=True)
m_cone = Metric(x**2, (x,))

print("Metric g₁₁(x) = x²")
print("Christoffel symbol Γ¹₁₁ =", m_cone.christoffel_sym)
print("√det(g) =", m_cone.sqrt_det_expr)

In [ ]:
# Solve a geodesic starting at x0 = 2.0 with velocity v0 = 1.0
geod = geodesic_solver(m_cone, 2.0, 1.0, (0, 3), n_steps=200)

# Initial vector: unit vector perpendicular to the geodesic (v_perp = 1.0)
# In 1D, "perpendicular" is not defined – we use a scalar. Actually parallel transport
# in 1D simply multiplies the scalar by a factor depending on the geometry.
# To make it interesting, we choose an initial vector v0 = 2.5.
initial_vec = 2.5
result = parallel_transport(m_cone, geod, initial_vec)

# Plot the transported vector as a function of time
plt.figure(figsize=(8, 5))
plt.plot(result['t'], result['v'], 'b-', label='Transported vector')
plt.axhline(initial_vec, color='r', linestyle='--', label='Initial value (flat comparison)')
plt.xlabel('t')
plt.ylabel('v(t)')
plt.title('Parallel transport along a geodesic on the cone metric')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
t_curve = np.linspace(0, 2, 200)
x_curve = 2 + t_curve  # linear motion
curve = {'t': t_curve, 'x': x_curve}

result2 = parallel_transport(m_cone, curve, 1.0)

plt.figure(figsize=(8, 5))
plt.plot(result2['t'], result2['v'], 'g-', label='Transported vector')
plt.xlabel('t')
plt.ylabel('v(t)')
plt.title('Parallel transport along a non‑geodesic curve')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## Riemannian Gradient and Hessian of a Scalar Function

In [ ]:
# Define a function with non‑constant gradient and Hessian on the cone
f_expr = x**4

grad_expr = m_cone.riemannian_gradient(f_expr)
hess_expr = m_cone.riemannian_hessian(f_expr)

print("f(x) =", f_expr)
print("Riemannian gradient ∇f =", grad_expr)
print("Riemannian Hessian ∇²f =", hess_expr)

# Numerical evaluation
x_vals = np.linspace(0.5, 3, 200)
grad_func = sp.lambdify(x, grad_expr, 'numpy')
hess_func = sp.lambdify(x, hess_expr, 'numpy')

grad_vals = grad_func(x_vals)   # now returns an array
hess_vals = hess_func(x_vals)   # now returns an array

# Ordinary derivatives for comparison
ord_grad = 4*x**3               # f'(x)
ord_hess = 12*x**2              # f''(x)

ord_grad_func = sp.lambdify(x, ord_grad, 'numpy')
ord_hess_func = sp.lambdify(x, ord_hess, 'numpy')
ord_grad_vals = ord_grad_func(x_vals)
ord_hess_vals = ord_hess_func(x_vals)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(x_vals, ord_grad_vals, 'r--', label="Ordinary derivative")
ax1.plot(x_vals, grad_vals, 'b-', label="Riemannian gradient")
ax1.set_xlabel('x')
ax1.set_ylabel('∇f')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(x_vals, ord_hess_vals, 'r--', label="Ordinary second derivative")
ax2.plot(x_vals, hess_vals, 'b-', label="Riemannian Hessian")
ax2.set_xlabel('x')
ax2.set_ylabel('∇²f')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('Comparison of ordinary vs. Riemannian derivatives on the cone metric')
plt.tight_layout()
plt.show()

## Gradient Descent on a Riemannian Manifold using Parallel Transport

In [ ]:
# Define potential V(x) = (x - 2)²
V_expr = (x - 2)**2
gradV_expr = m_cone.riemannian_gradient(V_expr)

# Numerical functions
gradV_func = sp.lambdify(x, gradV_expr, 'numpy')
V_func = sp.lambdify(x, V_expr, 'numpy')

# Initial point
x0 = 0.5
learning_rate = 0.3
n_steps = 20
t_step = 0.5  # integration time for each geodesic step

xs = [x0]
vs = [0.0]  # initial velocity

for _ in range(n_steps):
    x_curr = xs[-1]
    # Compute negative gradient at current point
    v_step = -learning_rate * gradV_func(x_curr)
    # Solve geodesic for a short time
    traj = geodesic_solver(m_cone, x_curr, v_step, (0, t_step), n_steps=10)
    # New point is the end of the geodesic
    x_new = traj['x'][-1]
    xs.append(x_new)
    vs.append(v_step)

xs = np.array(xs)
values = V_func(xs)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(xs, 'o-', label='Riemannian gradient descent')
plt.xlabel('Iteration')
plt.ylabel('x')
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.semilogy(values, 's-')
plt.xlabel('Iteration')
plt.ylabel('V(x)')
plt.grid(alpha=0.3)
plt.title('Convergence')
plt.tight_layout()
plt.show()

In [ ]:
# Ordinary gradient descent
x_eucl = [0.5]
for _ in range(n_steps):
    x_curr = x_eucl[-1]
    grad_ord = 2*(x_curr - 2)  # derivative of (x-2)²
    x_new = x_curr - learning_rate * grad_ord
    x_eucl.append(x_new)
x_eucl = np.array(x_eucl)

plt.plot(xs, 'o-', label='Riemannian')
plt.plot(x_eucl, 's-', label='Euclidean')
plt.xlabel('Iteration')
plt.ylabel('x')
plt.legend()
plt.grid(alpha=0.3)
plt.show()